# 00 — Build the AML images

**Run this before `e2e_austria_aml.ipynb`.** It also doubles as the reference for *which files an
image needs and why* — the thing that is easy to get wrong because one folder used to do three
unrelated jobs.

## Which image do you actually need?

fsd runs two node images. **They are independent and are rebuilt on different schedules** — Part A
and Part B below are separate on purpose.

| image | used by | rebuild when |
|---|---|---|
| **`fsd-aml-env`** (Part A) | download shards, datacube builds, `create_training_data`'s flatten | the **fsd source** changed |
| **`fsd-infer-sklearn`** (Part B) | the `run_inference` fan-out, `verify_image`'s smoke job | the fsd source changed, **or** your model's runtime deps changed |

> ⚠️ **Every run of a register cell creates a NEW version, even if nothing changed.**
> `az ml environment create` has no no-op: AML auto-increments unconditionally. That is
> deliberate — a version can never mutate under a run that already referenced it — but it means
> **you should only run the register cell of the part you actually need.** Each part starts with a
> status cell that tells you whether anything changed since you last registered.

**First time here?** Run Part 0, then Part A, then Part B, then Part C.
**Coming back?** Run Part 0, then only the part that changed, then Part C.

### You do NOT need to rebuild anything when

- you **retrained your model** — it rides in the bundle
- you **edited your adapter code** — it rides in the bundle too (spec 44)

Only a change to fsd itself, or to your dependency *family* (sklearn → torch), needs an image.

## Which files go where, and why

Three folders, one job each. They used to be one folder, which is why the wheel's role was
unclear.

```
notebooks/
  images/
    base/                    <- BUILD CONTEXT for fsd-aml-env      (Part A)
      Dockerfile                 what to install
      .dockerignore              keeps stray artifacts out of the context
      environment.yml            the AML asset definition (no version: AML auto-increments)
      fsd-0.1.0-*.whl            <- staged by Part A; gitignored
    sklearn/                 <- BUILD CONTEXT for fsd-infer-sklearn (Part B)
      Dockerfile   .dockerignore   environment.yml   fsd-0.1.0-*.whl
  demo_model/                <- YOUR MODEL. No Docker files, no wheel.
      my_adapter.py              -> bundle.save(code=[...])
      rf.joblib                  -> bundle.save(artifacts={"model": ...})
```

### Why the wheel sits beside the Dockerfile and *not* beside the model

This is the question the old layout could not answer, because `demo_model/` held the adapter, the
trained model, the Dockerfile **and** the wheel all at once.

- **`bundle.save` never reads the wheel.** Bundling takes your adapter source (`code=`) and your
  trained artifacts (`artifacts=`). That is all. A wheel in the model folder does nothing.
- **The wheel is what gets installed into the image.** `COPY fsd-*.whl /tmp/` only works if the
  wheel is inside the Docker *build context* — Docker cannot read a file outside it. That is the
  only reason the wheel has to be in a specific folder at all.
- **`verify_image(build_context=...)` re-reads that same folder later.** It opens the wheel and
  checks whether it carries `manifest_code_files` (spec 44). If it does not, the registered image
  was built from a pre-spec-44 fsd, the node would raise `ModuleNotFoundError` however good your
  bundle is, and you find that out on the driver in ~2 s instead of after a cold start.

So: `build_context` means *"the folder I built this image from"*, and it must still hold the wheel
afterwards. It is not, and never was, a model directory.

> ⚠️ **Since spec 47 Part D, `verify_image` RAISES `ValueError` if `build_context` holds no
> `fsd-*.whl`** — it no longer returns `pass: False`. A missing wheel is a mistake in your call,
> not a verdict about the image, and the two must not look alike. Practical consequence: **do not
> clean the wheel out of `images/sklearn/` after building.**

---

# Part 0 — Setup

**Always run this part.** It configures, builds the fsd wheel once, and defines the helpers
Parts A and B use. It registers nothing.

## 0.1 — Prerequisites

**Azure CLI**, logged in, with the `ml` extension:

```bash
az login                       # run in a terminal, not here — it opens a browser
az extension add -n ml         # once per machine
```

**Your Azure coordinates in `env.local.sh`.** This notebook is in a public repo, so it holds no
resource group, workspace, subscription id or URL. It reads them from the repo's existing
gitignored config file:

```bash
cp env.example.sh env.local.sh     # from the repo root; env.local.sh is gitignored
$EDITOR env.local.sh               # fill AZ_RG and AZ_ML_WORKSPACE at minimum
```

Every variable is documented in `docs/reference/environment.md`. Concrete values come from your
platform admin — this repo never carries them.

`az login` is interactive, so run it in a terminal. The cell below only *checks* the result.

In [ ]:
# Which subscription am I about to build into? It must be the one holding AZ_ML_WORKSPACE
# below -- `az ml environment create` has no subscription flag here, it uses this one.
#
# Prints a yes/no rather than the id: this notebook is committed, and a saved output
# carrying a subscription id or a user principal is exactly the leak that keeps notebooks
# out of public repos. Run `az account show` in a terminal when you need the detail.
import subprocess as _sp

_acct = _sp.run(["az", "account", "show", "--query", "id", "-o", "tsv"],
                capture_output=True, text=True)
print("az logged in:", bool(_acct.stdout.strip()))
if not _acct.stdout.strip():
    print("  ->", _acct.stderr.strip()[:300] or "run `az login` in a terminal")

## 0.2 — Configure

The only cell in this notebook you edit.

In [ ]:
import glob, hashlib, json, os, pathlib, re, shutil, subprocess, zipfile
from datetime import datetime, timezone

# --- local paths -------------------------------------------------------------------
def _find_repo():
    """The fsd checkout, found by marker rather than by `..`.

    A bare `Path("..")` is silently wrong the moment the kernel's cwd is not `notebooks/`
    -- it resolves to the parent of wherever you happen to be and every path below it is
    then subtly off, with no error until something far away fails.
    """
    for d in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (d / "pyproject.toml").exists() and (d / "src" / "fsd").is_dir():
            return d
    raise RuntimeError(
        f"no fsd checkout above {pathlib.Path.cwd()} -- run this notebook from inside the repo."
    )


FSD_REPO    = _find_repo()                          # the checkout the wheel is built from
NOTEBOOKS   = FSD_REPO / "notebooks"
BASE_CTX    = NOTEBOOKS / "images" / "base"
SKLEARN_CTX = NOTEBOOKS / "images" / "sklearn"


def env_local(path=None):
    """Read `env.local.sh` -- the repo's existing place for real Azure names (spec 41 D7).

    This notebook is in a PUBLIC repo, so it must not carry a resource group, a workspace
    name, a subscription id or a URL. `env.example.sh` is the tracked template with every
    value blank; `env.local.sh` is your filled-in copy and is gitignored. Parsed rather
    than sourced: no shell is spawned, and derived `$(...)` entries are skipped.
    """
    path = pathlib.Path(path or FSD_REPO / "env.local.sh")
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Create it once:\n"
            f"    cp {FSD_REPO / 'env.example.sh'} {path}\n"
            f"    $EDITOR {path}      # fill AZ_RG and AZ_ML_WORKSPACE at minimum\n"
            "It is gitignored. Concrete values for your platform come from your admin, "
            "never from this repo."
        )
    out = {}
    for line in path.read_text().splitlines():
        # `env.example.sh` writes `export AZ_RG=''   # what it means`, so a trailing
        # comment must be allowed -- requiring end-of-line right after the value made
        # EVERY line fail to match and reported a filled-in file as empty.
        m = re.match(
            r"""\s*export\s+(\w+)=(?:"([^"]*)"|'([^']*)'|([^#\s]*))\s*(?:\#.*)?$""", line)
        if not m:
            continue
        value = next((g for g in m.groups()[1:] if g is not None), "")
        if "$(" in value or "${" in value:      # derived entries -- not ours to resolve
            continue
        if value:
            out[m.group(1)] = value
    return out


_env = env_local()


def need(key, default=None):
    v = _env.get(key) or default
    if not v:
        raise KeyError(
            f"{key} is empty in env.local.sh -- fill it in. See docs/reference/environment.md."
        )
    return v


# --- Azure coordinates: the only genuinely private values, so the only ones read from
#     env.local.sh. Everything below is generic and safe to sit in a public repo. -------
AZ_RG           = need("AZ_RG")
AZ_ML_WORKSPACE = need("AZ_ML_WORKSPACE")

# --- image names. Deliberately NOT read from env.local.sh: an existing file there may
#     still name a pre-spec-44 inference image (`fsd-infer-env`, one image per adapter),
#     and silently registering into that name would rebuild the wrong thing. Edit here if
#     your platform uses different names.
#     Versions are NOT set: AML auto-increments on every register.
AZ_ENV_NAME       = "fsd-aml-env"          # Part A: download / datacube / flatten
AZ_INFER_ENV_NAME = "fsd-infer-sklearn"    # Part B: the above + scikit-learn + joblib

print("resource group :", AZ_RG)
print("workspace      :", AZ_ML_WORKSPACE)
print("part A image   :", AZ_ENV_NAME)
print("part B image   :", AZ_INFER_ENV_NAME)
print("base ctx       :", BASE_CTX)
print("sklearn ctx    :", SKLEARN_CTX)

## 0.3 — Build the fsd wheel

**The wheel is built from your working tree, not from a release.** Whatever is checked out and
uncommitted right now is what lands on the nodes. Confirm the commit before building — a stale
checkout here is the single most common cause of "the fix I just made isn't on the cluster".

In [ ]:
# What exactly am I about to package? Commit the fix first if this isn't what you expect.
!git -C {FSD_REPO} log --oneline -3
!git -C {FSD_REPO} status --porcelain

In [ ]:
# Built ONCE into a staging folder. Parts A and B each copy it into their own build
# context, so both images normally carry the SAME fsd -- see the warning in Part B if you
# ever let them diverge. --no-deps: we want the fsd wheel alone; the Dockerfiles resolve
# its dependencies inside the image.
STAGE = NOTEBOOKS / "images" / ".wheel"
shutil.rmtree(STAGE, ignore_errors=True)
STAGE.mkdir(parents=True)

_build = subprocess.run(
    [str(FSD_REPO / ".venv/bin/pip"), "wheel", str(FSD_REPO), "--no-deps", "-w", str(STAGE)],
    capture_output=True, text=True,
)
print(_build.stdout[-1500:], _build.stderr[-1500:])
_build.check_returncode()

WHEEL = pathlib.Path(glob.glob(str(STAGE / "fsd-*.whl"))[0])
print("\nwheel:", WHEEL.name)

In [ ]:
def wheel_digest(path):
    """A content hash that ignores zip metadata.

    `pip wheel` stamps timestamps, so two wheels built from identical source are never
    byte-identical -- hashing the file would report "changed" on every run and make the
    status cells useless. Hashing the sorted (member name, CRC) pairs compares what is
    actually INSIDE the wheel.
    """
    with zipfile.ZipFile(path) as zf:
        payload = "\n".join(f"{i.filename}:{i.CRC}" for i in sorted(zf.infolist(),
                                                                    key=lambda i: i.filename))
    return hashlib.sha256(payload.encode()).hexdigest()[:16]


# Only these paths end up inside an image: the package itself, its build metadata, and
# the build contexts. A scratch file anywhere else is irrelevant -- checking the WHOLE
# tree meant one stray untracked note pinned every status cell to "dirty" forever, which
# is exactly the signal those cells exist to give.
_IMAGE_INPUTS = ["src", "pyproject.toml", "notebooks/images"]


def git_state():
    rev = subprocess.run(["git", "-C", str(FSD_REPO), "rev-parse", "--short", "HEAD"],
                         capture_output=True, text=True).stdout.strip()
    dirty = bool(subprocess.run(
        ["git", "-C", str(FSD_REPO), "status", "--porcelain", "--", *_IMAGE_INPUTS],
        capture_output=True, text=True).stdout.strip())
    return rev + ("-dirty" if dirty else "")


WHEEL_DIGEST = wheel_digest(WHEEL)
GIT_STATE = git_state()

# Does this wheel carry spec 44's `manifest_code_files`? If False, `verify_image` will
# (correctly) reject every image built from it -- rebuild from a current checkout.
with zipfile.ZipFile(WHEEL) as zf:
    HAS_SPEC44 = "def manifest_code_files" in zf.read("fsd/model/bundle.py").decode()

print(f"wheel          : {WHEEL.name}")
print(f"content digest : {WHEEL_DIGEST}")
print(f"built from     : {GIT_STATE}")
print(f"has spec44     : {HAS_SPEC44}")
assert HAS_SPEC44, "this wheel predates spec 44 -- verify_image would reject any image built from it"

## 0.4 — Helpers

Nothing to decide here; run it and move on. These are what Parts A and B call.

`.last_registered.json` in each build context records what you last put in that image
(version, wheel digest, git state, timestamp). It is purely local bookkeeping — AML cannot tell
you what a registered version was built from, so this is how the status cells answer
*"has anything actually changed?"*.

In [ ]:
RECORD_NAME = ".last_registered.json"


def read_record(ctx):
    p = ctx / RECORD_NAME
    return json.loads(p.read_text()) if p.exists() else None


def write_record(ctx, name, version):
    (ctx / RECORD_NAME).write_text(json.dumps({
        "environment": name, "version": version, "wheel_digest": WHEEL_DIGEST,
        "git_state": GIT_STATE, "registered_at": datetime.now(timezone.utc).isoformat(),
    }, indent=2))


def latest_registered(name):
    """The highest version AML currently holds for `name`, or None if it has never been
    registered. Proves the ASSET exists -- never that its image finished building."""
    out = subprocess.run(
        ["az", "ml", "environment", "list", "-n", name, "-g", AZ_RG, "-w", AZ_ML_WORKSPACE,
         "--query", "[].version", "-o", "tsv"],
        capture_output=True, text=True,
    )
    versions = [v for v in out.stdout.split() if v.isdigit()]
    return max(versions, key=int) if versions else None


def status(ctx, name):
    """Do I need to run this part's register cell? Prints; decides nothing for you.

    Keyed on GIT STATE, not on the wheel digest. The Dockerfile and every fsd source file
    are tracked, so any change to what goes into this image either moves HEAD or makes the
    tree dirty -- that is a deterministic signal. The wheel digest is carried alongside as
    diagnostics (it answers "are my two images on the same fsd?") but is NOT the test:
    `pip wheel` is not guaranteed to be byte-reproducible, and a false "unchanged" is the
    one answer this cell must never give.
    """
    rec, live = read_record(ctx), latest_registered(name)
    print(f"{name}")
    print(f"  latest registered in AML : {live or '<never registered>'}")
    if rec is None:
        print(f"  last registered from here: <no local record>")
        print(f"  -> no record of what {name} was built from. Register if you are unsure.")
        return
    print(f"  last registered from here: {name}:{rec['version']} ({rec['registered_at'][:19]}Z)")
    print(f"      git {rec['git_state']}   wheel {rec['wheel_digest']}")
    print(f"  your checkout now        : git {GIT_STATE}   wheel {WHEEL_DIGEST}")
    if GIT_STATE.endswith("-dirty"):
        print(f"  -> CHANGED (uncommitted edits under {_IMAGE_INPUTS}). Register.")
    elif rec["git_state"] != GIT_STATE:
        print(f"  -> CHANGED ({rec['git_state']} -> {GIT_STATE}). Run the register cell.")
    else:
        print(f"  -> UNCHANGED. SKIP the register cell; keep using {name}:{rec['version']}.")


def register(ctx, name):
    """Register one environment and return the version AML assigned.

    Guarded on purpose: `v = !az ...` cannot fail -- a broken az silently yields its error
    text and every later cell then uses a garbage version. Seen live 2026-07-29:
    `built fsd-aml-env:No module named 'rpds.rpds'`.
    """
    for stale in glob.glob(str(ctx / "fsd-*.whl")):
        os.remove(stale)                      # never leave two: `ls fsd-*.whl` picks one
    shutil.copy2(WHEEL, ctx)

    out = subprocess.run(
        ["az", "ml", "environment", "create", "-f", str(ctx / "environment.yml"),
         "-g", AZ_RG, "-w", AZ_ML_WORKSPACE, "--query", "version", "-o", "tsv"],
        capture_output=True, text=True, cwd=ctx,
    )
    version = out.stdout.strip()
    if not version.isdigit():
        raise RuntimeError(
            f"{name}: az returned {version!r} (stderr: {out.stderr.strip()[:400]}) -- not a "
            "version number. STOP; see Troubleshooting at the bottom."
        )
    write_record(ctx, name, version)
    print(f"registered {name}:{version}   (wheel {WHEEL_DIGEST}, git {GIT_STATE})")
    return version


# Studio's `wsid` query parameter IS the workspace's ARM resource id -- ask az for it rather
# than string-building a path that would silently 404 on a typo.
_ws = subprocess.run(
    ["az", "ml", "workspace", "show", "-n", AZ_ML_WORKSPACE, "-g", AZ_RG,
     "--query", "id", "-o", "tsv"], capture_output=True, text=True,
)
WSID = _ws.stdout.strip()
if not WSID.startswith("/subscriptions/"):
    raise RuntimeError(f"could not resolve the workspace id: {WSID!r} / {_ws.stderr.strip()[:300]}")
_TENANT = subprocess.run(["az", "account", "show", "--query", "tenantId", "-o", "tsv"],
                         capture_output=True, text=True).stdout.strip()


def build_link(name, version):
    """Studio page for one environment version. THE ONLY WAY to see build status: an AML v2
    image build is an ACR task run, not an AML job, so no `az ml job list` query can see it
    (an earlier version of this notebook polled for one and printed `0/0` forever)."""
    from IPython.display import Markdown, display

    tid = f"&tid={_TENANT}" if _TENANT else ""
    url = f"https://ml.azure.com/environments/{name}/version/{version}?wsid={WSID}{tid}"
    display(Markdown(
        f"**Wait for `Build status: Succeeded` before using it** — ~10-20 min of ACR time, "
        f"occasionally flaky. Build log is on the page.\n\n- [`{name}:{version}`]({url})"
    ))


print("helpers ready")

---

# Part A — `fsd-aml-env` (general purpose)

Download shards, datacube builds, `create_training_data`'s flatten. **Rebuild only when the fsd
source changed.** Most users touching only their own model never rebuild this after the first
time.

Context: [`images/base/Dockerfile`](images/base/Dockerfile).

In [ ]:
status(BASE_CTX, AZ_ENV_NAME)

### A.2 — Register (only if A.1 said CHANGED)

**Running this creates a new version whether or not anything changed.** If A.1 said UNCHANGED,
skip to Part B and keep using the version it printed.

In [ ]:
AZ_ENV_VERSION = register(BASE_CTX, AZ_ENV_NAME)
build_link(AZ_ENV_NAME, AZ_ENV_VERSION)

---

# Part B — `fsd-infer-sklearn` (inference)

The `run_inference` fan-out and `verify_image`'s smoke job. Generic per **dependency family**,
never per model (spec 44): the adapter's source rides inside the bundle, so this image copies no
adapter and sets no `PYTHONPATH`. Every sklearn model you ever bundle runs on this one image.

**Rebuild when** the fsd source changed, **or** when your model's runtime deps changed. For a
different family (torch, xgboost) copy `images/sklearn/` to `images/<family>/`, swap the last two
deps in its `Dockerfile`, change `name:` in its `environment.yml`, and point this part at it.

Context: [`images/sklearn/Dockerfile`](images/sklearn/Dockerfile).

> **Keep the two images on the same fsd where you can.** Rebuilding only this one from a newer
> checkout leaves your datacubes built by one fsd and your inference run by another. That is
> usually fine — the artifacts on disk are the contract — but it is the first thing to check if a
> cube and a model disagree. A.1's `wheel digest` and B.1's should normally match.

In [ ]:
status(SKLEARN_CTX, AZ_INFER_ENV_NAME)

### B.2 — Register (only if B.1 said CHANGED)

**Running this creates a new version whether or not anything changed.**

In [ ]:
AZ_INFER_ENV_VERSION = register(SKLEARN_CTX, AZ_INFER_ENV_NAME)
build_link(AZ_INFER_ENV_NAME, AZ_INFER_ENV_VERSION)

---

# Part C — Carry the versions into the e2e notebook

Works whichever parts you ran: for anything you skipped it falls back to the local record, then
to whatever AML currently holds.

In [ ]:
def resolve(name, ctx, just_built):
    if just_built:
        return just_built, "registered just now"
    rec = read_record(ctx)
    if rec:
        return rec["version"], f"from the local record ({rec['registered_at'][:10]})"
    live = latest_registered(name)
    if live:
        return live, "latest registered in AML (no local record)"
    raise RuntimeError(f"{name} has never been registered -- run its Part above.")


_env, _env_why = resolve(AZ_ENV_NAME, BASE_CTX, globals().get("AZ_ENV_VERSION"))
_inf, _inf_why = resolve(AZ_INFER_ENV_NAME, SKLEARN_CTX, globals().get("AZ_INFER_ENV_VERSION"))

print("# paste into e2e_austria_aml.ipynb's config cell")
print(f'AZ_ENV_NAME          = "{AZ_ENV_NAME}"')
print(f'AZ_ENV_VERSION       = "{_env}"      # {_env_why}')
print(f'AZ_INFER_ENV_NAME    = "{AZ_INFER_ENV_NAME}"')
print(f'AZ_INFER_ENV_VERSION = "{_inf}"      # {_inf_why}')
print(f'INFER_BUILD_CONTEXT  = "./images/sklearn"')
print()
print("The wheel must stay in images/sklearn -- verify_image raises without it (spec 47 D11).")
print("Both image builds must read Succeeded in Studio before you run the e2e notebook.")

## Troubleshooting

**The version keeps going up even though nothing changed.** That is AML, not a bug:
`az ml environment create` always mints a new version and has no no-op mode. It is deliberate —
a version can never mutate under a run that already referenced it. The fix is to *not run the
register cell*: each part's status cell tells you whether anything changed.

**`az` returned something that isn't a version.** `az ml` tries to auto-upgrade itself and cannot
on an AML compute instance — the `ml` extension there is installed system-wide at
`/opt/az/extensions/ml`, owned by root, while you run as `azureuser`. It fails with
`Permission denied` and leaves the extension **half-deleted**, after which every `az ml` command
breaks. Run these steps from your laptop, or reinstall the extension with `sudo`.

**Where is the build status?** Only in Studio — the link each register cell prints. An AML v2
environment build is an **ACR task run, not an AML job**, so nothing in `az ml job list` shows it,
and the v2 `Environment` object carries no build state at all. `az ml environment show` proves the
*asset* is registered, never that the image is built.

**`verify_image` says `wheel_has_spec44: False`.** The registered image was built from a
pre-spec-44 fsd. Re-run Part 0 then the relevant part, from a current checkout.

**`verify_image` raises `ValueError: build_context=... contains no fsd-*.whl`.** The wheel was
cleaned out of `images/sklearn/`. Re-run Part 0 and Part B.2; do not delete it afterwards.

**A build failed.** Studio → Environments → the version → build log. Re-running the register cell
mints a fresh version, which is cheap and never mutates the old one.

**This notebook is committed — keep it clean.** Before `git add`, run *Kernel → Restart & Clear
All Outputs*. Its saved outputs would otherwise carry your subscription id, tenant id, workspace
URL and local paths into a public repo. `tests/test_notebooks.py` fails the build if outputs or
hardcoded identifiers get in, so a leak cannot land silently — but clearing them yourself is
faster than finding out from a red test.